# Initial-margin CVaR MKABSDE

This notebook displays the retained call and butterfly campaigns. It reads the generated JSON files and contains no solver implementation.

## Complete retraining command

Run from the repository root:

```bash
python -m initial_margin.run_campaign \
  --product all \
  --steps 50 100 200 \
  --seeds 42 43 44 \
  --paths 200000 \
  --degree 3 \
  --margin-period 0.02 \
  --device cuda
```

The paper defaults use 24 Gauss--Legendre nodes for the call tail and 64 nodes for the complete butterfly law.

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('repository root not found')
result_root = root / 'initial_margin' / 'results'
products = ('call', 'butterfly')

## Control-variate prices

The direct column targets the anticipated equation. The non-linear and linear columns target the two proxy BSDEs.

In [2]:
for product in products:
    payload = json.loads((result_root / f'atm_{product}_experiment.json').read_text())
    article_price = 2.1959 if product == 'call' else 0.3112
    lines = [
        f'### ATM {product}',
        '',
        '| $N$ | direct MKABSDE | non-linear proxy | linear proxy |',
        '|---:|---:|---:|---:|',
    ]
    for row in payload['summary']:
        lines.append(
            f"| {row['time_steps']} | {row['direct_mkabsde_control_variate_price_mean']:.6f} | "
            f"{row['nonlinear_proxy_control_variate_price_mean']:.6f} | "
            f"{row['linear_proxy_control_variate_price_mean']:.6f} |"
        )
    lines.extend(['', f'Article proxy benchmark: **{article_price:.4f}**.'])
    display(Markdown('\n'.join(lines)))

### ATM call

| $N$ | direct MKABSDE | non-linear proxy | linear proxy |
|---:|---:|---:|---:|
| 50 | 2.192040 | 2.196548 | 2.196013 |
| 100 | 2.191933 | 2.196453 | 2.195940 |
| 200 | 2.191931 | 2.196439 | 2.195922 |

Article proxy benchmark: **2.1959**.

### ATM butterfly

| $N$ | direct MKABSDE | non-linear proxy | linear proxy |
|---:|---:|---:|---:|
| 50 | 0.310357 | 0.310243 | 0.311270 |
| 100 | 0.310363 | 0.310241 | 0.311294 |
| 200 | 0.310353 | 0.310231 | 0.311303 |

Article proxy benchmark: **0.3112**.

## Interpretation

The proxy equations are no longer anticipated, so the backward implementation reduces to the classical implicit BSDE scheme with Hermite conditional regressions. The direct MKABSDE has no independent oracle in the source article. The three retained time grids show stability, but their small and non-monotone adjacent differences do not support a robust empirical $O(h)$ claim; regression and Monte Carlo noise may already dominate.